### 1. 모델 로드

In [1]:
import joblib

lda_model = joblib.load("lda_model.pkl")
dictionary = joblib.load("dictionary.pkl")

### 2. 전처리 함수

In [ ]:
import re
from kiwipiepy import Kiwi

# Kiwi 객체 (서버 시작 시 1번만 생성)
kiwi = Kiwi()

# 품사 필터
TARGET_POS = ["NNG", "NNP", "VV", "VA"]

# 불용어 통합
STOPWORDS = set([
    '기자','사진','뉴스','연합뉴스','뉴시스','보도','관련','이번','대한','통해','하다','이다',
    '지난','현재','이날','오전','오후','당시','이후','앞서','최근','정부','관계자','업계','측','대통령','국회','위원회',
    '등','것','수','때','위해','경우','때문','대해','지난해','올해','내년','년','월','일','억','만','천',
    '파이낸셜뉴스','KBS','한국경제','매일경제','이데일리','SBS Biz','부산일보','서울경제','머니투데이','세계일보','국제신문',
    '가능','상황','수준','계획','내용','포함','실제','방안','검토','일각','입장','통해',
    '발표','조사','응답','보고','설명','강조','언급','진단','요청','건의',
    '전체','가운데','이전','이후','처음','주요','전반','핵심','본격','적극','철저',
    '효과','역할','문제','요소','체계','기반','구성','구축','확보','강화','지원',
    '작년','새해','연내','시기','기간','차례','대상','이용','통하','위하','대하',
    '나타나','밝히','따르','보이','점치','내다보','늘리','줄이','높이','갖추','만들'
])

# 1. 텍스트 정제
def clean_text(text: str) -> str:
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^가-힣\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 2. 토큰화
def tokenize(text: str):
    tokens = []
    result = kiwi.analyze(text)[0][0]
    for token, pos, _, _ in result:
        if pos in TARGET_POS:
            tokens.append(token)
    return tokens

# 3️. 후처리 (불용어 + 길이 + 숫자)
def postprocess(tokens):
    return [
        t for t in tokens
        if t not in STOPWORDS
        and len(t) > 1
        and not t.isdigit()
    ]

# 최종 전처리 함수
def preprocess(text: str):
    text = clean_text(text)
    tokens = tokenize(text)
    tokens = postprocess(tokens)
    return tokens


### 3. 추론 코드

In [ ]:
def predict(text: str):
    tokens = preprocess(text)

    if not tokens:
        return {
            "error": "유효한 단어가 없습니다."
        }

    bow = dictionary.doc2bow(tokens)
    topics = lda_model.get_document_topics(bow)

    dominant_topic = max(topics, key=lambda x: x[1])[0]

    return {
        "topics": topics,
        "dominant_topic": dominant_topic
    }